# Fields to update
1. MB_ID (Done)
2. Country
3. Sub_genre
4. Homepage
5. Bandcamp page

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [ ]:
# Notion imports
from Notion.reader import get_artists_db
from Notion.database import myNotionDatabases
from Data.artists import Artist
from Data.tags import clean_tags
from Datasources.Last_Fm import get_artist_tags
# MusicBrainz imports
from Datasources.MusicBrainz import search_artist,get_artist_info

import random

In [ ]:
notion_response = get_artists_db()
artists_db = []
db = myNotionDatabases()
artist_database = {}

for artist in notion_response:
    extracted_artist = db.extract_artist(artist)
    artists_db.append(extracted_artist)
    key = extracted_artist['artist_name'].lower().strip()
    artist_database[key] = Artist(
        name=extracted_artist['artist_name'],
        mbid=extracted_artist['mb_id'],
        country=extracted_artist['country'],
        tags=extracted_artist.get('subgenre', []),
        website_url=extracted_artist.get('website'),
        bandcamp_url=extracted_artist.get('bandcamp_url')
    )

### Artists producing errors:
* Senses - Error - Exists under MB ID "9941e53e-601b-4fc3-9d08-6bc770c4c65c" I suspect its a confidence thing.
    * Not even a confidence thing, there are too many artists called the same thing. MB ID doesnt know what to do here.
    * Need to handle this better, ignore exact matchs if the results have more than 1 exact match.
    * Even after doing this its wrong, its returning "Senses Fail" totaly different band.
* Warduna - No match - Does not show up on MB at all.
    * Fine this will be manual.
* Hanabie - No Match - Seems to be an issue with the fact they are from japan. 
    * Might need to look for their sort name
    * Will keep manual for now.
* Windrose - No match
    * MB has them under "592b488d-dd42-4d9a-a343-d7ca4f69d0bb" but the name is "wind rose". Need to check what it should be from the bands sites.

In [ ]:
for artist in random.sample(sorted(artist_database.values(), key=lambda a: a.name), min(10, len(artist_database))): #Selecting X random artists for testing purposes
#for artist in artist_database: #Iterating through all artists in the database
    if artist.mbid is None:
        try:
            mb_id, match_result = search_artist(artist_name=artist.name, confidence_threshold=85)
            if mb_id is None:
                print(f"No match found for {artist.name} with the reason: {match_result}")
                continue
            
            mb_artist_info = get_artist_info(mb_id)
            artist.mbid = mb_id
            artist.country = mb_artist_info.get("country")
            artist.website_url = mb_artist_info.get("homepage")
            artist.bandcamp_url = mb_artist_info.get("bandcamp")
            artist.tags = mb_artist_info.get("tags", [])

            print(f'{artist.name} - {mb_id} - {match_result} - {mb_artist_info["country"].name} - {mb_artist_info["homepage"]} - {mb_artist_info["bandcamp"]} ')
        except Exception as e:
            print(f"Error processing {artist.name} - {mb_id}: {e}")

In [ ]:
#Get the Last.fm info for the artist.
last_fm_artists = []
for artist in artist_database.values():
    if artist.mbid is not None:
        last_fm_artists.append(artist)

In [ ]:
for artist in last_fm_artists:
    print(artist.name)
    last_fm_response = get_artist_tags(mb_id=artist.mbid)
    artist.add_tags(last_fm_response)

In [ ]:
for artist in last_fm_artists:
    cleaned_tags = clean_tags(artist.tags)
    sorted_tags = sorted(cleaned_tags, key=lambda x: x[1], reverse=True)
    Top_x = sorted_tags[:3]
    declared_subgenres = []

    for tag in Top_x:
        declared_subgenres.append(tag)
    artist.tags = Top_x

In [ ]:
#Commenting out for now to avoid damaging database with test runs. Uncomment to update mb_id in Notion.
#update_mb_id(artist["notion_id"], mb_id)